In [ ]:
import numpy as np
import torch
import torch.optim as optim
import logging
import matplotlib.pyplot as plt
from argparse import ArgumentParser
from torch.autograd import Variable
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.distributions import Normal, OneHotCategorical
import torch.nn.utils as nn_utils
import torch
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix
torch.manual_seed(123456)
np.random.seed(123456)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)        
class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians,n_hidden_layers,bn, dout, dout_value):
        super(MDN, self).__init__()        
        layers = [nn.Linear(41, n_hidden), nn.Tanh()] #Input features
        if bn == 1:
            layers.append(nn.BatchNorm1d(n_hidden))    
        for _ in range(n_hidden_layers - 2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())
        if dout==1:
            layers.append(nn.Dropout(dout_value))
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())   
        else:
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())       
        self.z_h = nn.Sequential(*layers)       
        self.z_pi = nn.Linear(n_hidden, n_gaussians)
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = nn.Linear(n_hidden, n_gaussians)  
    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))+ 1e-8
        sigma = torch.clamp(sigma, min=1e-4)
        mu = 0 + (1 - 0) * (torch.tanh(self.z_mu(z_h)) + 1) / 2
        return pi, sigma, mu    
oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0*np.pi) 
def gaussian_distribution(y, mu, sigma):
    result = (y.expand_as(mu) - mu) * torch.reciprocal(sigma)
    result = -0.5 * (result * result)
    return (torch.exp(result) * torch.reciprocal(sigma)) * oneDivSqrtTwoPI
def mdn_loss_fn(pi, sigma, mu, y):
    result = gaussian_distribution(y, mu, sigma) * pi
    result1 = torch.sum(result, dim=1)
    result2 = -torch.log(result1+1e-12)
    return torch.mean(result2)


model = MDN(n_hidden=91, n_gaussians=2,n_hidden_layers=6,bn=0, dout=0, dout_value=0.21738) #base BCC Ti-free
model.eval()
optimizer = optim.Adam(model.parameters(), lr=0.00007)
PATH = "checkpoint/model-812.pt" #should be change based on the lowest error 
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])


data_x = np.load('xlo_test_4.npy')[:,:] #Should be changed in each active learning cycle
data_y = np.load('ylo_test_4.npy') #Should be changed in each active learning cycle
#data_y[:,1]=data_y[:,1]+data_y[:,3] #Only with np.load('ylo_test.npy'), it should be acivated 
data_x_np = torch.tensor(data_x, dtype=torch.float)
data_y_np = torch.tensor(data_y, dtype=torch.float)[:,1:2]  
pi_variable, sigma_variable, mu_variable = model(data_x_np)
loss = mdn_loss_fn(pi_variable, sigma_variable, mu_variable, data_y_np)
weighted_average_mu = torch.sum(pi_variable * mu_variable, dim=-1)
weighted_sigma = torch.sqrt(torch.sum(pi_variable * (sigma_variable ** 2 + (mu_variable - weighted_average_mu.unsqueeze(-1)) ** 2), dim=-1))
weighted_average_mu_np = weighted_average_mu.detach().numpy()
weighted_sigma_np = weighted_sigma.detach().numpy()


n=np.arange(0,weighted_average_mu_np.shape[0],1).reshape(-1,1)
n=np.hstack([n,weighted_average_mu_np.reshape(-1,1),data_y_np]) 
n[:,1] = (n[:, 1] >= 0.9).astype(int)
n[:,2] = (n[:, 2] >= 0.9).astype(int)   
cm = confusion_matrix(n[:,2], n[:,1])
tn, fp, fn, tp = cm.ravel()  
# print('Precision=',precision_score(n[:,2], n[:,1]))
# print('Recall=',recall_score(n[:,2], n[:,1]))
# print('F1=',f1_score(n[:,2], n[:,1]))


def select_500_kmeans_bcc(Xt,n,uncertainty=False,B=500,tau=0.90,use_pca=False, pca_dims=20,random_state=42):  
    yt=n[:,1]
    C_idx = np.where(yt >= tau)[0]
    step = 0
    while C_idx.size < B and tau > 0.0:
        tau = max(0.0, tau - 0.02)
        C_idx = np.where(yt >= tau)[0]
        step += 1
    if C_idx.size < B:
        raise ValueError("Not enough candidates even after lowering threshold.")
    if uncertainty:
        m=np.zeros([0,3])
        for i in range(n.shape[0]):
            if yt[i]>=tau:
                m=np.append(m,n[[i],:],axis=0)
        m=m[np.argsort(m[:,2]),:]
        nm=3*B 
        C_idx=np.copy(m[:nm,0].astype('int')) #Should be changed as follows: ":nm" for high certain route and "-nm:" for low certain route   
    Xc = Xt[C_idx]
    if use_pca and Xc.shape[1] > pca_dims:
        pca = PCA(n_components=pca_dims, random_state=random_state).fit(Xc)
        Xemb = pca.transform(Xc)
    else:
        Xemb = Xc
    km = KMeans(n_clusters=B, n_init=10, random_state=random_state)
    km.fit(Xemb)
    centers = km.cluster_centers_
    labels = km.labels_
    picked = []
    for k in range(B):
        members = np.where(labels == k)[0]
        if members.size == 0:
            d_all = np.linalg.norm(Xemb - centers[k], axis=1)
            picked.append(int(np.argmin(d_all)))
            continue
        Xm = Xemb[members]
        d = np.linalg.norm(Xm - centers[k], axis=1)
        picked.append(members[int(np.argmin(d))])
    picked = np.array(picked, dtype=int)
    picked_unique = np.unique(picked)
    if picked_unique.size < B:
        remaining = np.setdiff1d(np.arange(len(C_idx)), picked_unique, assume_unique=True)
        need = B - picked_unique.size
        picked = np.concatenate([picked_unique, remaining[:need]])
    else:
        picked = picked_unique[:B]
    return C_idx[picked]


#--------Part 1--------------
# n=np.arange(0,weighted_average_mu_np.shape[0],1).reshape(-1,1)
# n=np.hstack([n,weighted_average_mu_np.reshape(-1,1),weighted_sigma_np.reshape(-1,1)]) 
# idx_500 = select_500_kmeans_bcc(data_x, n, uncertainty=False,B=500, tau=0.9, use_pca=False, pca_dims=20, random_state=42) # B should be changed (sample size)
#----------------------------

#--------Part 2--------------
n=np.arange(0,weighted_average_mu_np.shape[0],1).reshape(-1,1)
n=np.hstack([n,weighted_average_mu_np.reshape(-1,1),weighted_sigma_np.reshape(-1,1)]) 
idx_500 = select_500_kmeans_bcc(data_x, n,uncertainty=True,B=50, tau=0.9, use_pca=False, pca_dims=20, random_state=42) # B should be changed (sample size)
#----------------------------

Xadd=data_x[idx_500]
yadd=data_y[idx_500]
n=0
for i in range(yadd.shape[0]):
    if yadd[i,1]>=0.9:
        n=n+1
print('accuracy=',n/yadd.shape[0])
# all_idx = np.arange(len(data_x))
# test_idx = np.setdiff1d(all_idx, idx_500, assume_unique=True)
# X_test = data_x[test_idx]    
# y_test = data_y[test_idx]
# np.random.seed(42)
# num_rows=np.shape(Xadd)[0]
# permutation = np.random.permutation(num_rows)
# Xadd = Xadd[permutation]
# yadd = yadd[permutation]
# i=50 #sample size
# cycle=5 #next active learning cycle 
# with open('xlo_%d.npy' %cycle,'wb') as f:
#     np.save(f,Xadd[:int(0.9*i),:])
# with open('ylo_%d.npy' %cycle,'wb') as f:
#     np.save(f,yadd[:int(0.9*i),:]) 
# with open('xlo_val_%d.npy' %cycle,'wb') as f:
#     np.save(f,Xadd[int(0.9*i):,:])
# with open('ylo_val_%d.npy' %cycle,'wb') as f:
#     np.save(f,yadd[int(0.9*i):,:])   
# with open('xlo_test_%d.npy' %cycle,'wb') as f:
#     np.save(f,X_test)
# with open('ylo_test_%d.npy' %cycle,'wb') as f:
#     np.save(f,y_test) 